In [11]:
# Cell 1 — Imports and credentials
import os
import time
import httpx
import litellm
import pandas as pd
from datasets import load_dataset

os.environ["OPENAI_API_KEY"] = "not-needed"  # bypass deepeval's default provider check

LITELLM_URL = "https://litellm.mittelstand-zukunftslab.de"
LITELLM_KEY = "sk-pfrgLhvL2Idn3HCWC0yPAQ"        # same key you used in query_rag.py
QDRANT_URL = "https://qdrant.mittelstand-zukunftslab.de"
QDRANT_KEY = "zeuhzyfpmfyp7i71opfbglk9cwyg7uhd"          # same key you used in upload_direct.py
COLLECTION_NAME = "agrim_legal_bench_v1"

qdrant_session = httpx.Client(base_url=QDRANT_URL, headers={"api-key": QDRANT_KEY}, timeout=30)

In [14]:
# Cell 1b — Fix asyncio conflicts in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [12]:
# Cell 2 — Load the QA dataset
qa_dataset = load_dataset("isaacus/legal-rag-bench", "qa", split="test")
print(len(qa_dataset))

100


In [5]:
# Cell 3 — The retrieval+generation function (same logic as query_rag.py)
def answer_question(question, top_k=3):
    embed_response = litellm.embedding(
        model="openai/qwen/qwen3-embedding-8b",
        input=[question],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=30
    )
    query_vector = embed_response.data[0]["embedding"]

    search_resp = qdrant_session.post(
        f"/collections/{COLLECTION_NAME}/points/search",
        json={"vector": query_vector, "limit": top_k, "with_payload": True}
    )
    search_resp.raise_for_status()
    results = search_resp.json()["result"]
    context_chunks = [r["payload"].get("text", "") for r in results]
    context_str = "\n\n---\n\n".join(context_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
{context_str}

Question: {question}

Answer:"""

    chat_response = litellm.completion(
        model="openai/openrouter/qwen/qwen3.8-27b",
        messages=[{"role": "user", "content": prompt}],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=60
    )
    return {
        "context": context_chunks,
        "answer": chat_response.choices[0].message.content
    }

In [6]:
# Cell 4 — Quick sanity test on ONE question before running all 100
test_q = qa_dataset[0]["question"]
result = answer_question(test_q)
print(result["answer"])

The context does not contain the answer. It only states that a juror who knows someone involved in the trial should apply to be excused to prevent potential bias, but it does not specify whether the judge is required to grant the excusal.


In [7]:
# Cell 5 — DeepEval judge model wrapper
from deepeval.models.base_model import DeepEvalBaseLLM

class ProxyJudgeLLM(DeepEvalBaseLLM):
    def __init__(self, model, api_base, api_key):
        self.model, self.api_base, self.api_key = model, api_base, api_key

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        r = litellm.completion(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            api_key=self.api_key,
            api_base=self.api_base,
            timeout=60,
        )
        return r.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        r = await litellm.acompletion(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            api_key=self.api_key,
            api_base=self.api_base,
            timeout=60,
        )
        return r.choices[0].message.content

    def get_model_name(self):
        return self.model

judge = ProxyJudgeLLM("openai/openrouter/openai/gpt-4.1-mini", LITELLM_URL, LITELLM_KEY)

# Quick test — should print a short reply, not an error
print(judge.generate("Say OK"))

OK


In [8]:
# Cell 6 — Run the pipeline across all 100 QA pairs
from deepeval.test_case import LLMTestCase

test_cases = []
failed = []

for qa in qa_dataset:
    try:
        r = answer_question(qa["question"])
        test_cases.append(LLMTestCase(
            input=qa["question"],
            actual_output=r["answer"],
            expected_output=qa["answer"],
            retrieval_context=r["context"],
        ))
    except Exception as e:
        failed.append({"id": qa.get("id"), "error": str(e)})
    time.sleep(0.5)

print(f"Succeeded: {len(test_cases)} / {len(qa_dataset)}")
print(f"Failed: {len(failed)}")
if failed:
    print(failed[:5])

Succeeded: 100 / 100
Failed: 0


In [17]:
# Cell 7 — Define the DeepEval metrics (synchronous mode to avoid Jupyter asyncio conflicts)
from deepeval.metrics import FaithfulnessMetric, ContextualPrecisionMetric, GEval
from deepeval.test_case import LLMTestCaseParams

faithfulness = FaithfulnessMetric(threshold=0.80, model=judge, async_mode=False)
contextual_precision = ContextualPrecisionMetric(threshold=0.75, model=judge, async_mode=False)
answer_correctness = GEval(
    name="Answer Correctness",
    criteria="Determine whether the actual output is factually correct and complete relative to the expected output, without omitting key legal conclusions.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.75,
    model=judge,
    async_mode=False,
)

C:\Users\agrim\AppData\Local\Temp\ipykernel_10140\4219820706.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [19]:
# Cell 8 — Run evaluation (synchronous, using new AsyncConfig pattern)
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig

eval_results = evaluate(
    test_cases,
    metrics=[faithfulness, contextual_precision, answer_correctness],
    async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/openrouter/openai/gpt-4.1-mini, 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openai/openrouter/openai/gpt-4.1-mini, 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using 
openai/openrouter/openai/gpt-4.1-mini, strict=False, async_mode=False)...

Task was destroyed but it is pending!
task: <Task pending name='Task-8' coro=<_async_in_context.<locals>.run_in_context() running at 
C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\ipykernel\utils.py:60> wait_for=<Task 
pending name='Task-15' coro=<Kernel.shell_main() running at 
C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> 
cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at 
C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>

Task was destroyed but it is pending!
task: <Task pending name='Task-15' coro=<Kernel.shell_main() running at 
C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> 
cb=[Task.__wakeup()]>

C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\pydantic\main.py:263: RuntimeWarning: 
coroutine 'Kernel.shell_main' was never awaited
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
RuntimeWarning: Enable tracemalloc to get the object allocation traceback

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              Harry is serving as a juror in a burglary trial. He is also a professional           │
│  │                         locksmith. During the proceedings, Harry is shown a lock which, in his expert        │
│  │                         view, is too damaged to have been lockpicked in the manner described in court.       │
│  │                         Why might Harry's expert knowledge of lockpicking be irrelevant when assessing       │
│  │                         the physical evidence?                                                               │
│  │     Actual Output:      The provided context does not contain the answer to this question. It only           │
│  │                         addresses how a jury should evaluate and weigh expert witness testimony, not how     │
│  │                         a juror's own personal or professional expertise should be handled.                  │
│  │     Expected Output:    Judges are advised to “warn the jury of the possible dangers of conducting even      │
│  │                         simple experiments in the jury room” as “the present condition of the object         │
│  │                         experimented with may not be the same as its condition at the relevant time.”        │
│  │                         Harry can draw his own conclusions, but his view about the lock’s damage is          │
│  │                         irrelevant without additional evidence about how and when the lock's condition       │
│  │                         changed.                                                                             │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                     ┃ Score ┃ Threshold ┃ Reason                                          │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness               │ 1.00  │ 0.80      │ The score is 1.00 because there are no          │
│              │                            │       │           │ contradi...                                     │
│        PASS  │ Contextual Precision       │ 1.00  │ 0.75      │ The score is 1.00 because the first node in     │
│              │                            │       │           │ ret...                                          │
│        FAIL  │ Answer Correctness [GEval] │ 0.30  │ 0.75      │ The actual output correctly notes the absence   │
│              │                            │       │           │ of an explicit answer about jurors using        │
│              │                            │       │           │ their own expertise, aligning partially with    │
│              │                            │       │           │ the need to discuss the irrelevance of          │
│              │                            │       │      

⚠ WARNING: No hyperparameters logged.
» ]8;id=4412401;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2464.07s | token cost: None)
» Test Results (100 total tests):
   » Pass Rate: 35.0% | Passed: 35 | Failed: 65

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [20]:
# Cell 9 — Export detailed results to CSV
rows = []
for test_result in eval_results.test_results:
    row = {"input": test_result.input, "actual_output": test_result.actual_output}
    for metric_data in test_result.metrics_data:
        row[f"{metric_data.name}_score"] = metric_data.score
        row[f"{metric_data.name}_success"] = metric_data.success
        row[f"{metric_data.name}_reason"] = metric_data.reason
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("deepeval_results_legal_rag_bench.csv", index=False)
print(df[[c for c in df.columns if "_score" in c]].mean())

Faithfulness_score                  0.87675
Contextual Precision_score          0.71000
Answer Correctness [GEval]_score    0.59300
dtype: float64


In [21]:
low_correctness = df[df["Answer Correctness [GEval]_score"] < 0.5].sort_values("Answer Correctness [GEval]_score")
print(f"Number of low-scoring answers: {len(low_correctness)}")
print(low_correctness[["input", "actual_output", "Answer Correctness [GEval]_reason"]].head(10).to_string())

Number of low-scoring answers: 36
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  input                                                                                                                                                                                                                                                                                                                                                                                                               

In [22]:
test_case_ids = [11, 32, 28, 21]
for i in test_case_ids:
    q = qa_dataset[i]["question"]
    r_wide = answer_question(q, top_k=8)
    print(f"Q: {q[:80]}...")
    print(f"A (top_k=8): {r_wide['answer']}")
    print(f"Ground truth: {qa_dataset[i]['answer']}")
    print("---")

Q: Does Victoria have a law/laws protecting against double jeopardy?...
A (top_k=8): The context does not contain the answer.
Ground truth: Yes. The bench book states that “Unless expressly provided for by law, an accused must not be punished more than once for the same act or omission (Interpretation of Legislation Act 1984 (Vic) s 51; R v Sessions [1998] 2 VR 304).” This is often referred to as double jeopardy.
---
Q: Is the right to silence based solely on common law?...
A (top_k=8): No. While the context states that the right to remain silent when questioned by a person in authority is a "fundamental rule of the common law," it is not based solely on common law. The context also references statutory law, specifically section 89 of the *Evidence Act 2008*, which addresses the prohibition of unfavourable inferences from silence and notes there is "significant overlap between the scope of s 89 and the common law."
Ground truth: No. The bench book explains that “the right of silence ha

In [23]:
def diagnose_retrieval(qa_index, top_k=20):
    qa = qa_dataset[qa_index]
    question = qa["question"]
    correct_id = qa["relevant_passage_id"]

    embed_response = litellm.embedding(
        model="openai/qwen/qwen3-embedding-8b",
        input=[question],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=30
    )
    query_vector = embed_response.data[0]["embedding"]

    search_resp = qdrant_session.post(
        f"/collections/{COLLECTION_NAME}/points/search",
        json={"vector": query_vector, "limit": top_k, "with_payload": True}
    )
    results = search_resp.json()["result"]

    retrieved_ids = [r["payload"].get("id") for r in results]
    print(f"Question: {question[:80]}...")
    print(f"Correct passage ID: {correct_id}")
    print(f"Retrieved IDs (in rank order): {retrieved_ids}")
    if correct_id in retrieved_ids:
        print(f"FOUND at rank {retrieved_ids.index(correct_id) + 1}")
    else:
        print("NOT FOUND even in top 20")
    print("---")

for i in [11, 28, 21]:
    diagnose_retrieval(i)

Question: Does Victoria have a law/laws protecting against double jeopardy?...
Correct passage ID: 3.12-c1-s1
Retrieved IDs (in rank order): ['7.7.1-c14-s1', '5.9-c1-s1', '1.2-c2-s4', '1.5.1-c1-s1', '11.2-c4-s1', '9.4.1-c6-s1', '1.7-c4-s1', '7.4.12-c7-s1', '3.9-c3-s3', '9.1.2.1-c7-s1', 'B9.4.1-c6-s1', '9.1.2.3-c6-s1', '11.4-c6-s1', '7.3.23-c8-s1', '8.14.2-c5-s2', '1.7-c4-s15', '7.4.12-c6-s1', '5.4-c4-s4', '7.4.13-c6-s1', '3.10-c4-s1']
NOT FOUND even in top 20
---
Question: Chris is on trial for intentionally or recklessly causing a bushfire. His counse...
Correct passage ID: 4.13.1-c2-s1
Retrieved IDs (in rank order): ['7.5.21.1-c4-s1', '7.5.21-c2-s1', '7.5.21.1-c1-s1', '7.5.21.2-c1-s1', '7.5.21-c1-s1', '1.5.1-c1-s1', '1.7-c4-s1', '1.7-c4-s15', '1.6-c1-s1', '7.2.2.1-c6-s1', '7.2.2.2-c6-s1', '7.5.19.1-c5-s1', '7.5.19.1-c1-s1', '8.5-c3-s1', '3.9-c1-s13', '1.5.1-c1-s3', '7.5.21.1-c3-s1', '11.1-c8-s1', '1.7-c4-s12', '8.1-c13-s1']
NOT FOUND even in top 20
---
Question: The defence raises th

In [24]:
def show_top_result_text(qa_index):
    qa = qa_dataset[qa_index]
    question = qa["question"]

    embed_response = litellm.embedding(
        model="openai/qwen/qwen3-embedding-8b",
        input=[question],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=30
    )
    query_vector = embed_response.data[0]["embedding"]

    search_resp = qdrant_session.post(
        f"/collections/{COLLECTION_NAME}/points/search",
        json={"vector": query_vector, "limit": 1, "with_payload": True}
    )
    top = search_resp.json()["result"][0]
    print(f"Question: {question[:100]}")
    print(f"Top retrieved ID: {top['payload'].get('id')}")
    print(f"Top retrieved TEXT: {top['payload'].get('text')[:400]}")
    print("---")

for i in [11, 28, 21]:
    show_top_result_text(i)

Question: Does Victoria have a law/laws protecting against double jeopardy?
Top retrieved ID: 7.7.1-c14-s1
Top retrieved TEXT: ## Duplicity and Multiple Offences
---
Question: Chris is on trial for intentionally or recklessly causing a bushfire. His counsel says he was carryi
Top retrieved ID: 7.5.21.1-c4-s1
Top retrieved TEXT: ## Summary

**To summarise**, before you can find NOA guilty of intentionally or recklessly causing a bushfire, the prosecution must prove to you, beyond reasonable doubt:

**One** – That the accused caused a fire;

**Two** – That the accused caused the fire intentionally or recklessly;

**Three** – That the accused was reckless as to the spread of the fire to vegetation on property belonging to a
---
Question: The defence raises the issue of mental impairment and bears the onus of proving it. There is a psych
Top retrieved ID: 8.4-c4-s1
Top retrieved TEXT: ## Establishing Mental Impairment



1. A person is presumed not to be suffering from a mental impairment 

In [25]:
# Cell 10 — Summary of Sprint 1 DeepEval baseline
summary = """
SPRINT 1 DEEPEVAL BASELINE — legal-rag-bench (100 QA pairs)

Faithfulness:        0.877  (threshold 0.80)  -> PASS
Contextual Precision: 0.710  (threshold 0.75)  -> FAIL (close)
Answer Correctness:   0.593  (threshold 0.75)  -> FAIL

Root cause investigation (36 low-scoring cases analyzed):
- Model does NOT hallucinate; it correctly says "context does not contain
  the answer" when retrieval fails, which is why Faithfulness stays high.
- Root cause is retrieval precision on multi-hop / cross-domain questions:
  the correct passage was NOT in top-20 retrieved results for several
  failing cases (e.g. double jeopardy, s78A evidence exception, mental
  impairment witness rule).
- Retrieved chunks were topically adjacent but not the precise passage
  (e.g. "Duplicity" retrieved instead of "Double Jeopardy"; bushfire
  charge elements retrieved instead of the cross-referenced Evidence Act
  exception embedded within a bushfire scenario).
- This matches legal-rag-bench's stated design as a "reasoning-intensive"
  benchmark requiring multi-hop retrieval across disconnected sections.

Candidate fixes for future sprints (not implemented yet):
- Hybrid BM25 + vector search
- Query rewriting / decomposition for multi-hop questions
- Reranking step after initial top-k retrieval
- Increasing top_k (tested up to 8, only partial improvement)
"""
print(summary)

with open("sprint1_deepeval_findings.txt", "w") as f:
    f.write(summary)


SPRINT 1 DEEPEVAL BASELINE — legal-rag-bench (100 QA pairs)

Faithfulness:        0.877  (threshold 0.80)  -> PASS
Contextual Precision: 0.710  (threshold 0.75)  -> FAIL (close)
Answer Correctness:   0.593  (threshold 0.75)  -> FAIL

Root cause investigation (36 low-scoring cases analyzed):
- Model does NOT hallucinate; it correctly says "context does not contain
  the answer" when retrieval fails, which is why Faithfulness stays high.
- Root cause is retrieval precision on multi-hop / cross-domain questions:
  the correct passage was NOT in top-20 retrieved results for several
  failing cases (e.g. double jeopardy, s78A evidence exception, mental
  impairment witness rule).
- Retrieved chunks were topically adjacent but not the precise passage
  (e.g. "Duplicity" retrieved instead of "Double Jeopardy"; bushfire
  charge elements retrieved instead of the cross-referenced Evidence Act
  exception embedded within a bushfire scenario).
- This matches legal-rag-bench's stated design as a 